# Cardiometabolic Phenotype Discovery with Unsupervised Learning

**NHANES 2011–2018 · PCA + K-Means vs. Autoencoder + K-Means**

**Goal:** Explore biomarker-based cardiometabolic subgroups and compare linear and nonlinear representations for clustering. These groups are *exploratory profiles*, not validated clinical diagnoses or patient-specific risk predictions.

**Reproducibility:** Upload a ZIP containing `DATASETS/Separate Datasets/` with the NHANES `.xpt` files for cycles G–J. Run cells from top to bottom in Google Colab. Metrics below are from a prior executed run; re-running may produce slightly different autoencoder results unless the environment is identical.

**Contributors:** Add the names of all project contributors before publication.

## 1. Environment and data upload

In [ ]:
%pip -q install pandas pyreadstat scikit-learn matplotlib tensorflow

from pathlib import Path
from functools import reduce
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from google.colab import files
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

uploaded = files.upload()  # Upload the NHANES project ZIP
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Please upload a .zip containing DATASETS/Separate Datasets/*.xpt')
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall('/content/nhanes_project')  # Use trusted ZIP files only
matches = list(Path('/content/nhanes_project').rglob('Separate Datasets'))
if not matches:
    raise FileNotFoundError('No DATASETS/Separate Datasets folder found inside ZIP')
RAW_DIR = matches[0]
print('NHANES data folder:', RAW_DIR)

## 2. Load and merge four NHANES cycles

In [ ]:
CYCLES = {'G': '2011–2012', 'H': '2013–2014', 'I': '2015–2016', 'J': '2017–2018'}
TABLES = ['DEMO', 'GLU', 'GHB', 'HDL', 'TRIGLY', 'TCHOL', 'BIOPRO', 'BPX', 'BMX']

cycle_frames = []
for suffix, period in CYCLES.items():
    tables = []
    for prefix in TABLES:
        candidates = list(RAW_DIR.glob(f'{prefix}_{suffix}.[xX][pP][tT]'))
        if not candidates:
            raise FileNotFoundError(f'Missing NHANES file: {prefix}_{suffix}.xpt')
        tables.append(pd.read_sas(candidates[0]))
    merged = reduce(lambda left, right: left.merge(right, on='SEQN', how='inner'), tables)
    merged['Cycle'] = period
    cycle_frames.append(merged)

all_data = pd.concat(cycle_frames, ignore_index=True)
print('Merged dataset:', all_data.shape)
print('Cycles:', all_data['Cycle'].value_counts().to_dict())

## 3. Prepare eight biomarkers

In [ ]:
BIOMARKERS = {
    'LBXGLU': 'Glucose', 'LBXGH': 'HbA1c', 'LBDHDD': 'HDL cholesterol',
    'LBXTR': 'Triglycerides', 'LBXTC': 'Total cholesterol',
    'BMXBMI': 'BMI', 'BPXSY1': 'Systolic BP', 'BPXDI1': 'Diastolic BP'
}
features = all_data[list(BIOMARKERS)].dropna().copy().reset_index(drop=True)
print('Complete cases:', len(features), 'of', len(all_data))
display(features.describe().round(2))

scaler = StandardScaler()
X = scaler.fit_transform(features)
print('Standardized shape:', X.shape)

**Missing-data limitation:** Complete-case analysis excludes respondents with missing values and may introduce selection bias. NHANES survey weights and complex sampling design are not incorporated; cluster frequencies should not be interpreted as population prevalence.

## 4. PCA + K-Means

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)
pca_model = KMeans(n_clusters=3, random_state=SEED, n_init=10)
pca_labels = pca_model.fit_predict(X_pca)
print('Explained variance ratio:', pca.explained_variance_ratio_.round(3))

pca_means = features.assign(cluster=pca_labels).groupby('cluster')[list(BIOMARKERS)].mean()
display(pca_means.round(2))
print('Cluster sizes:', pd.Series(pca_labels).value_counts().sort_index().to_dict())

**Interpretation:** The previous run identified a lower-biomarker-risk profile (cluster 0), an intermediate profile (cluster 2), and a profile with markedly elevated glucose and HbA1c (cluster 1). We avoid adding unstandardized biomarker values into an arbitrary composite risk score.

## 5. Choosing K and visualizing PCA clusters

In [ ]:
def plot_k_diagnostics(embedding, title):
    ks = list(range(2, 10))
    inertia, silhouettes = [], []
    for k in ks:
        model = KMeans(n_clusters=k, random_state=SEED, n_init=10)
        predicted = model.fit_predict(embedding)
        inertia.append(model.inertia_)
        silhouettes.append(silhouette_score(embedding, predicted))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(ks, inertia, marker='o')
    axes[0].set(title=f'{title}: elbow', xlabel='Number of clusters', ylabel='Inertia')
    axes[1].plot(ks, silhouettes, marker='o')
    axes[1].set(title=f'{title}: silhouette', xlabel='Number of clusters', ylabel='Silhouette score')
    plt.tight_layout(); plt.show()
    return pd.DataFrame({'k': ks, 'inertia': inertia, 'silhouette': silhouettes})

pca_k_diagnostics = plot_k_diagnostics(X_pca, 'PCA')

plt.figure(figsize=(9, 6))
for cluster in sorted(np.unique(pca_labels)):
    subset = X_pca[pca_labels == cluster]
    plt.scatter(subset[:, 0], subset[:, 1], s=8, alpha=0.45, label=f'Cluster {cluster}')
plt.scatter(*pca_model.cluster_centers_.T, s=180, c='black', marker='X', label='Centroids')
plt.title('PCA + K-Means: exploratory cardiometabolic clusters')
plt.xlabel('Principal component 1'); plt.ylabel('Principal component 2')
plt.legend(); plt.show()

## 6. Train a nonlinear autoencoder

In [ ]:
inputs = tf.keras.Input(shape=(X.shape[1],))
h = tf.keras.layers.Dense(16, activation='relu')(inputs)
h = tf.keras.layers.Dense(8, activation='relu')(h)
latent = tf.keras.layers.Dense(3, activation='linear', name='latent_space')(h)
h = tf.keras.layers.Dense(8, activation='relu')(latent)
h = tf.keras.layers.Dense(16, activation='relu')(h)
outputs = tf.keras.layers.Dense(X.shape[1], activation='linear')(h)
autoencoder = tf.keras.Model(inputs, outputs)
encoder = tf.keras.Model(inputs, latent)
autoencoder.compile(optimizer='adam', loss='mse')
history = autoencoder.fit(X, X, epochs=50, batch_size=256, validation_split=0.2, verbose=0)
print('Final training loss:', round(history.history['loss'][-1], 4))
print('Final validation loss:', round(history.history['val_loss'][-1], 4))
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Train MSE')
plt.plot(history.history['val_loss'], label='Validation MSE')
plt.xlabel('Epoch'); plt.ylabel('Reconstruction MSE')
plt.title('Autoencoder learning curves'); plt.legend(); plt.show()
X_ae = encoder.predict(X, verbose=0)
print('Latent shape:', X_ae.shape)

## 7. Autoencoder + K-Means

In [ ]:
ae_k_diagnostics = plot_k_diagnostics(X_ae, 'Autoencoder')
ae_model = KMeans(n_clusters=3, random_state=SEED, n_init=10)
ae_labels = ae_model.fit_predict(X_ae)
ae_means = features.assign(cluster=ae_labels).groupby('cluster')[list(BIOMARKERS)].mean()
display(ae_means.round(2))
print('Cluster sizes:', pd.Series(ae_labels).value_counts().sort_index().to_dict())

plt.figure(figsize=(9, 6))
for cluster in sorted(np.unique(ae_labels)):
    subset = X_ae[ae_labels == cluster]
    plt.scatter(subset[:, 0], subset[:, 1], s=8, alpha=0.45, label=f'Cluster {cluster}')
plt.scatter(*ae_model.cluster_centers_[:, :2].T, s=180, c='black', marker='X', label='Centroids')
plt.title('Autoencoder + K-Means (first two latent dimensions)')
plt.xlabel('Latent dimension 1'); plt.ylabel('Latent dimension 2')
plt.legend(); plt.show()

**Important:** Cluster IDs are arbitrary and may change between runs. Interpret them using the biomarker summaries above; do not reuse a hardcoded mapping from another run. In the supplied execution, autoencoder cluster **2** had the highest glucose and HbA1c averages, cluster **1** was intermediate, and cluster **0** was lower risk.

## 8. Compare clustering quality

In [ ]:
comparison = pd.DataFrame([
    {'Method': 'PCA + K-Means', 'Silhouette': silhouette_score(X_pca, pca_labels),
     'Davies–Bouldin': davies_bouldin_score(X_pca, pca_labels)},
    {'Method': 'Autoencoder + K-Means', 'Silhouette': silhouette_score(X_ae, ae_labels),
     'Davies–Bouldin': davies_bouldin_score(X_ae, ae_labels)}
])
display(comparison.round(4))

### Previously observed results (from the supplied executed Colab notebook)

| Method | Silhouette ↑ | Davies–Bouldin ↓ |
|---|---:|---:|
| PCA + K-Means | **0.3849** | **0.8982** |
| Autoencoder + K-Means | 0.3406 | 1.1052 |

On this run, PCA yielded better separation by both internal metrics. The nonlinear representation did not improve clustering quality under the tested architecture and hyperparameters. These internal metrics do not validate clinical utility, and comparisons across different embedding spaces should be interpreted cautiously.

## 9. Conclusions and limitations

- **Finding:** Both pipelines identified heterogeneous biomarker profiles; PCA + K-Means scored better on the reported clustering metrics.
- **No clinical validation:** Risk-like labels are descriptive, not diagnoses or prognostic scores.
- **Model selection:** `k=3` was used for comparison; the diagnostic plots should be inspected rather than treating three clusters as clinically established.
- **Generalizability:** Complete-case filtering and the omission of NHANES sampling weights limit population-level interpretation.
- **Stability:** Autoencoder results may vary across runs; assess robustness across seeds, sampling schemes, and model settings in future work.
- **Data provenance:** Data originate from publicly available NHANES survey cycles G–J (2011–2018).